# IRIS World Model Tutorial

This notebook demonstrates how to train and perform inference with the IRIS (Implicit Reinforcement using Image-based State Abstractions) world model from TorchWM.

## What is IRIS?

IRIS is a sample-efficient RL agent that uses:

- **Vector Quantized VAE (VQ-VAE)**: For learning discrete latent representations of images
- **Transformer**: For modeling latent dynamics and predicting future states
- **Actor-Critic**: For learning policies from imagined rollouts in latent space

IRIS is particularly effective for the Atari 100k benchmark (100,000 environment steps).

In [ ]:
# Setup and Imports
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import cv2

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# TorchWM imports
from world_models.models.iris_agent import IRISAgent
from world_models.configs.iris_config import IRISConfig
from world_models.inference.operators import IrisOperator
from world_models.envs.ale_atari_env import make_atari_env
from world_models.memory.iris_memory import IRISReplayBuffer
from world_models.training.train_iris import IRISTrainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Configuration

The `IRISConfig` class provides all hyperparameters for training IRIS.

In [ ]:
# Create IRIS configuration
config = IRISConfig()

# Customize for quick demo
config.total_epochs = 3  # Short for demo
config.env_steps_per_epoch = 500
config.training_steps_per_epoch = 50
config.autoencoder_batch_size = 32
config.transformer_batch_size = 8
config.actor_critic_batch_size = 16
config.start_autoencoder_after = 0
config.start_transformer_after = 1
config.start_actor_critic_after = 1
config.eval_episodes = 2
config.eval_interval = 2
config.checkpoint_interval = 10

print("=== IRIS Configuration ===")
print(f"Total epochs: {config.total_epochs}")
print(f"Environment steps per epoch: {config.env_steps_per_epoch}")
print(f"Transformer timesteps: {config.transformer_timesteps}")
print(f"Embedding dim: {config.embedding_dim}")
print(f"Num tokens: {config.num_tokens}")
print(f" Imagination horizon: {config.imagination_horizon}")

## Creating the IRIS Agent

The IRIS agent consists of an autoencoder, transformer, and actor-critic modules.

In [ ]:
# Create Atari environment to get action space
game = "ALE/Pong-v5"
env = make_atari_env(game, obs_type="rgb", frameskip=4, max_episode_steps=27000)
action_size = env.action_space.n
print(f"Game: {game}")
print(f"Action space size: {action_size}")

# Create IRIS agent
agent = IRISAgent(
    config=config,
    action_size=action_size,
    device=device
)
print("\nIRIS Agent created successfully!")
print(f"\nAutoencoder: VQ-VAE with {config.codebook_size} codebook entries")
print(f"Transformer: {config.transformer_num_layers} layers, {config.transformer_hidden_dim} hidden dim")

## Training IRIS

Training involves four phases:
1. **Data Collection**: Gather experience from the environment
2. **Autoencoder Training**: Learn to reconstruct observations
3. **Transformer Training**: Learn latent dynamics
4. **Actor-Critic Training**: Learn policy via imagined rollouts

We'll use the IRISTrainer for full training with evaluation.

In [ ]:
# Create trainer and train
trainer = IRISTrainer(
    game=game,
    device=str(device),
    seed=42,
    config=config
)

print("Starting IRIS training...")
print("=" * 50)

# Train the model
metrics = trainer.train(
    total_epochs=config.total_epochs,
    eval_interval=config.eval_interval,
    save_dir="checkpoints/iris/notebook"
)

print("\n" + "=" * 50)
print("Training complete!")

In [ ]:
# Plot training metrics
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

if 'collection_return' in metrics:
    axes[0, 0].plot(metrics['collection_return'])
    axes[0, 0].set_title('Collection Return')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Return')

if 'recon_loss' in metrics:
    axes[0, 1].plot(metrics['recon_loss'])
    axes[0, 1].set_title('Reconstruction Loss')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')

if 'token_loss' in metrics:
    axes[1, 0].plot(metrics['token_loss'])
    axes[1, 0].set_title('Token Prediction Loss')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Loss')

if 'actor_loss' in metrics:
    axes[1, 1].plot(metrics['actor_loss'])
    axes[1, 1].set_title('Actor Loss')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Loss')

plt.tight_layout()
plt.savefig('iris_training_metrics.png', dpi=150)
plt.show()

print("Training metrics saved to iris_training_metrics.png")

## Evaluating the Trained Model

Let's evaluate the trained model on the environment.

In [ ]:
# Evaluate the trained agent
print("Evaluating trained agent...")

def preprocess_frame(frame, size=64):
    """Preprocess frame for IRIS."""
    frame = cv2.resize(frame, (size, size), interpolation=cv2.INTER_LINEAR)
    frame = frame.astype(np.float32) / 255.0
    return frame.transpose(2, 0, 1)  # CHW format

# Run evaluation episodes
eval_returns = []

for ep in range(3):
    obs, _ = env.reset()
    obs = preprocess_frame(obs)
    
    episode_return = 0
    done = False
    steps = 0
    max_steps = 500
    
    while not done and steps < max_steps:
        frame_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
        action = agent.act(frame_tensor, epsilon=0.0, temperature=0.5).item()
        
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        obs = preprocess_frame(obs) if not done else obs
        episode_return += reward
        steps += 1
    
    eval_returns.append(episode_return)
    print(f"Episode {ep + 1}: Return = {episode_return}, Steps = {steps}")

print(f"\nMean evaluation return: {np.mean(eval_returns):.2f} +/- {np.std(eval_returns):.2f}")

## Inference with IRIS Operator

Use the `IrisOperator` for convenient preprocessing during inference.

In [ ]:
# Using the IrisOperator
from PIL import Image

iris_operator = IrisOperator(image_size=64, action_dim=action_size)

# Example preprocessing
dummy_image = Image.new('RGB', (64, 64), color='blue')
dummy_action = 0

processed = iris_operator.process({'image': dummy_image, 'action': dummy_action})
print("Processed observation shape:", processed['obs'].shape)
print("Processed action shape:", processed['action'].shape)
print("Observation value range:", processed['obs'].min().item(), "to", processed['obs'].max().item())

## Model Components

Let's explore the IRIS agent components.

In [ ]:
# Explore model architecture
print("=== IRIS Agent Components ===")
print(f"\nAutoencoder (VQ-VAE):")
print(f"  - Embedding dim: {config.embedding_dim}")
print(f"  - Num tokens: {config.num_tokens}")
print(f"  - Codebook size: {config.codebook_size}")

print(f"\nTransformer:")
print(f"  - Timesteps: {config.transformer_timesteps}")
print(f"  - Num heads: {config.transformer_num_heads}")
print(f"  - Hidden dim: {config.transformer_hidden_dim}")
print(f"  - Num layers: {config.transformer_num_layers}")

print(f"\nActor-Critic:")
print(f"  - Imagination horizon: {config.imagination_horizon}")
print(f"  - Action dim: {action_size}")

print(f"\nEncoder (VQ-VAE):")
print(f"  - Type: {type(agent.encoder).__name__}")

print(f"\nTransformer:")
print(f"  - Type: {type(agent.transformer).__name__}")

print(f"\nActor-Critic:")
print(f"  - Actor type: {type(agent.actor).__name__}")
print(f"  - Critic type: {type(agent.critic).__name__}")

## Saving and Loading

IRIS agents can be saved and loaded for later use.

In [ ]:
# Save checkpoint
save_path = "checkpoints/iris/notebook_final.pt"
agent.save(save_path)
print(f"Model saved to {save_path}")

# Load checkpoint
new_agent = IRISAgent(config=config, action_size=action_size, device=device)
new_agent.load(save_path)
new_agent.eval()
print("Model loaded successfully!")

# Test the loaded model
obs, _ = env.reset()
obs = preprocess_frame(obs)
frame_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
action = new_agent.act(frame_tensor, epsilon=0.0).item()
print(f"Action from loaded model: {action}")

## Summary

In this tutorial, you learned:

1. **Configuration**: How to set up `IRISConfig` with customizable hyperparameters
2. **Agent Creation**: How to create an IRIS agent with autoencoder, transformer, and actor-critic
3. **Training**: How to train using IRISTrainer with all four phases
4. **Evaluation**: How to evaluate and visualize results
5. **Inference**: How to use `IrisOperator` and run the agent in Atari environments
6. **Saving/Loading**: How to save and load model checkpoints